# EDA — PhiUSIIL phishing URL dataset

Experimentation only: every function called here (`build_dataset`, `shortcut_audit`,
`extract_feature_frame`, ...) lives in `copilot_ml` and is unit-tested there. This notebook
does not contain any logic of its own — it only calls and displays.

Run `uv run copilot-ml data build` first so `ml/datasets/processed/*.parquet` exists.

In [ ]:
import pandas as pd

from copilot_ml.config import MLSettings
from copilot_ml.data.audit import shortcut_audit
from copilot_ml.features.schema import HOST_FEATURES
from copilot_ml.features.vectorize import extract_feature_frame

pd.set_option("display.width", 120)
settings = MLSettings.from_env()
train = pd.read_parquet(settings.processed_dir / "train.parquet")
train.shape, train["label"].mean()

## Class balance

In [ ]:
train["label"].value_counts(normalize=True).rename({0: "legitimate", 1: "phishing"})

## Shortcut audit (full detail behind the summary in the data card)

See `ml/reports/data_card.md` §5 for the generated version of this table with narrative.

In [ ]:
pd.DataFrame([vars(i) | {"class_exclusive": i.class_exclusive} for i in shortcut_audit(train)])

## Host-only feature distributions by class

These are the features the `host` view trains on (M1.4 ablation) — none of them depend on
path/query/scheme, so they should separate the classes less dramatically than the shortcut
audit above, but (if the model is to be useful) still meaningfully.

In [ ]:
sample = train.sample(n=min(20_000, len(train)), random_state=42)
features = extract_feature_frame(sample["url"])
features["label"] = sample["label"].to_numpy()

In [ ]:
numeric_host_features = [f for f in HOST_FEATURES if f != "host_tld"]
features.groupby("label")[numeric_host_features].mean().T.rename(
    columns={0: "legitimate_mean", 1: "phishing_mean"}
)

In [ ]:
features.groupby("label")["host_tld"].value_counts(normalize=True).groupby(level=0).head(8)

## Correlated / near-duplicate host features

Sanity check before training: flag pairs of numeric host features that are almost perfectly
correlated, since that would mean the feature set has redundant columns.

In [ ]:
import numpy as np

corr = features[numeric_host_features].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
high_pairs = upper.stack().sort_values(ascending=False)
high_pairs[high_pairs > 0.85]